# 大数定律的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.8/simulate_LLN.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：模拟 Bernoulli(0.5) 样本的累积均值，展示样本均值随样本量增大收敛于总体均值0.5。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 模拟**大数定律**：样本均值随样本量增大而收敛于总体均值。代码要点：
- `local M=5000` 设定最大样本量；生成5000个 Bernoulli(0.5) 样本（`gen u=runiform()`、`gen x=u>=0.5` 是生成0/1随机变量的简便写法，其均值为0.5）；
- 预设 `gen x_bar=.` 存放“前 i 个观测的均值”，用 `forvalues i=1/5000` 循环：每轮 `su x if _n<=i` 对前 i 个观测求均值，`replace x_bar=r(mean) if _n==i` 把它写到第 i 行——于是 x_bar 构成**累积样本均值序列**；
- `gen n=_n` 为横坐标，`line x_bar n, yline(0.5)` 画累积均值随样本量的变化，并用水平参考线标出总体均值0.5。图中曲线在初期波动很大，随后迅速收敛到0.5附近——直观展示了大数定律。
（注：循环中每次都对前 i 个观测重复求均值，是 O(n²) 的教学写法；目的是让“累积均值”的定义一目了然。）


In [ ]:
%%stata
clear
set more off
set seed 8855
local M=5000
set obs `M'
gen u=runiform()
gen x=u>=0.5
gen x_bar=.
forvalues i=1/`M'{
	qui{
		su x if _n<=`i'
		replace x_bar = r(mean) if _n==`i'
	}
}
gen n=_n
line x_bar n, yline(0.5)
